# Graph-Based RAG — Knowledge Graphs for Retrieval-Augmented Generation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/graph_rag.ipynb)

> **Goal:** Move beyond flat vector search by building a **knowledge graph** from text,
> then combining graph traversal with vector similarity for richer, multi-hop retrieval.

**What you will learn in this notebook:**
1. Why vector-only RAG misses structural relationships (cause–effect, part-of, temporal).
2. Knowledge graph fundamentals: nodes, edges, triples `(subject, predicate, object)`.
3. LLM-driven entity and relationship extraction — no NER library needed.
4. Building and visualising a knowledge graph with `networkx`.
5. Graph-based retrieval: subgraph extraction from query entities.
6. Hybrid retrieval: graph traversal **+** FAISS vector search.
7. Multi-hop reasoning over the graph.
8. Community detection to discover entity clusters.
9. A complete Graph-RAG pipeline: extract → build → retrieve → generate.
10. Head-to-head comparison: vector-only vs. graph-augmented retrieval.

**Stack:** `sentence-transformers (bge-base-en-v1.5)`, `FAISS`, `transformers + bitsandbytes (Qwen3-8B 4-bit)`, `networkx`, `numpy`.  
No LangChain, no LlamaIndex, no OpenAI API — everything runs locally.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Limitations of Vector-Only RAG**
- Understand **What Is a Knowledge Graph?**
- Understand **Environment Setup**
- Understand **Synthetic Knowledge Base**
- Understand **Knowledge Graph Construction**


<div style="text-align: center;">

<img src="../images/graph_rag.svg" alt="Graph RAG" style="width:100%; height:auto;">
</div>

## 1 — Limitations of Vector-Only RAG

Standard RAG works as follows: chunk documents → embed → store in a vector DB → at
query time, embed the query and retrieve the top-k most similar chunks → feed them to
an LLM for answer generation.

This is powerful for **semantic similarity**, but it is blind to **structural
relationships**:

| Relationship Type | Example | Vector RAG Handles It? |
|---|---|---|
| **Cause → Effect** | "Deforestation → increased CO₂" | ❌ Chunks may not co-occur |
| **Part-of / Hierarchy** | "Mitochondria are part of eukaryotic cells" | ❌ Only if nearby in text |
| **Temporal / Sequential** | "Event A happened before Event B" | ❌ Ordering is lost |
| **Multi-hop** | "Who founded the company that acquired X?" | ❌ Requires chaining facts |

**Key insight:** Vector similarity finds *what sounds related*, but a knowledge graph
captures *how things are actually connected*. Graph-RAG combines both signals.

## 2 — What Is a Knowledge Graph?

A **knowledge graph** (KG) is a directed, labelled graph where:

- **Nodes** (entities): People, organisations, concepts, events, places.
- **Edges** (relationships): Directed connections with a label describing the relationship.
- **Properties**: Optional key-value metadata on nodes or edges.

The atomic unit of a KG is a **triple**:

```
(subject, predicate, object)
```

For example:
- `(Albert Einstein, won, Nobel Prize in Physics)`
- `(Tesla, acquired_by, Elon Musk)`  — wait, that's wrong! Musk *leads* Tesla.
- `(Elon Musk, CEO_of, Tesla)`

**Why triples?** They are the simplest structure that preserves directional
relationships. A collection of triples forms a graph that machines can traverse,
query (SPARQL), and reason over.

Famous KGs: Google Knowledge Graph, Wikidata, DBpedia, YAGO.

### From Text to KG

Traditionally, KG construction required:
1. **Named Entity Recognition (NER)** — find entity spans in text.
2. **Relation Extraction (RE)** — classify the relationship between entity pairs.

With modern LLMs, we can do *both* in a single prompted call, which is the approach
we take in this notebook.

## 3 — Environment Setup

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy networkx matplotlib

In [ ]:
import os, pathlib

# Google Drive cache for large models
CACHE = "/content/drive/MyDrive/models"
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    CACHE = str(pathlib.Path.home() / ".cache" / "huggingface" / "hub")

os.makedirs(CACHE, exist_ok=True)
os.environ["HF_HOME"]           = CACHE
os.environ["TRANSFORMERS_CACHE"] = CACHE
os.environ["HF_HUB_CACHE"]      = CACHE
print(f"Model cache → {CACHE}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# ── Embedding model ──────────────────────────────────────────────
EMB_ID = "BAAI/bge-base-en-v1.5"
embedder = SentenceTransformer(EMB_ID, cache_folder=CACHE)
print(f"Embedder loaded: {EMB_ID}  dim={embedder.get_sentence_embedding_dimension()}")

# ── LLM: Qwen3-8B  4-bit NF4 ───────────────────────────────────
LLM_ID = "Qwen/Qwen3-8B"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE)
model = AutoModelForCausalLM.from_pretrained(
    LLM_ID, quantization_config=bnb, device_map="auto", cache_dir=CACHE,
)
print(f"LLM loaded: {LLM_ID}  (4-bit NF4)")

In [ ]:
def generate(prompt, max_new_tokens=512, temperature=0.7):
    """Generate a response from Qwen3-8B with /no_think for deterministic output."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": prompt + "\n/no_think"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.9, do_sample=True,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Quick smoke test
print(generate("Say hello in one sentence.", max_new_tokens=30))

## 4 — Synthetic Knowledge Base

We create a small but richly connected corpus about a fictional tech ecosystem.
The text is designed to contain explicit entities and relationships that vector
search alone would struggle to chain together.

The corpus covers:
- **Companies:** NovaTech, QuantumLeap, AeroSys, DataPrime, GreenGrid
- **People:** CEO/CTO roles, founders, board members
- **Products & Technologies:** AI chips, cloud platforms, autonomous drones
- **Events:** Acquisitions, partnerships, product launches

This is intentionally structured so that answering certain questions requires
traversing multiple hops (e.g., *"Who is the CEO of the company that acquired DataPrime?"*).

In [ ]:
DOCUMENTS = [
    # ── Companies & People ────────────────────────────────────────
    "NovaTech is an artificial intelligence company headquartered in San Francisco. "
    "It was founded in 2015 by Dr. Elena Marchetti, who serves as CEO. NovaTech "
    "specialises in large language models and AI accelerator chips.",

    "QuantumLeap is a quantum computing startup based in Boston. Its CEO is Dr. Raj "
    "Patel, a former MIT professor. QuantumLeap developed the Q-100 processor, the "
    "first commercially viable 100-qubit chip.",

    "AeroSys designs autonomous drones for agricultural monitoring. The company was "
    "founded in 2018 by Carlos Rivera in Austin, Texas. Its flagship product is the "
    "SkyScout drone, which uses computer vision for crop health analysis.",

    "DataPrime is a cloud data analytics platform founded by Sarah Chen in 2016. "
    "It provides real-time data pipelines, ETL services, and a managed data lake. "
    "DataPrime is headquartered in Seattle.",

    "GreenGrid is a renewable energy technology company based in Denver. It was founded "
    "by Dr. Amir Hassan in 2014. GreenGrid manufactures smart grid controllers and "
    "solar panel optimisation software.",

    # ── Relationships & Events ─────────────────────────────────────
    "In January 2024, NovaTech acquired DataPrime for $2.3 billion. The acquisition "
    "aimed to integrate DataPrime's real-time data pipelines with NovaTech's AI models, "
    "creating an end-to-end AI-data platform.",

    "NovaTech and QuantumLeap announced a strategic partnership in March 2024 to develop "
    "quantum-enhanced AI training. Under the deal, NovaTech will provide training "
    "datasets and QuantumLeap will supply quantum processing units.",

    "AeroSys signed a contract with GreenGrid in February 2024 to use SkyScout drones "
    "for inspecting solar farms. GreenGrid's smart grid controllers will relay drone "
    "telemetry data in real time.",

    "Dr. Elena Marchetti joined the board of directors of GreenGrid in April 2024, "
    "strengthening the ties between NovaTech and the renewable energy sector. She also "
    "holds an advisory role at QuantumLeap.",

    "Sarah Chen, founder of DataPrime, became VP of Data Engineering at NovaTech after "
    "the acquisition. She now reports directly to Dr. Elena Marchetti.",

    # ── Products & Technology ──────────────────────────────────────
    "NovaTech's flagship product is the NeuroCore AI chip, which delivers 500 TOPS "
    "(tera operations per second) at only 75 watts. The NeuroCore chip is manufactured "
    "by TSMC using a 3nm process.",

    "The Q-100 processor by QuantumLeap operates at 15 millikelvin and achieves quantum "
    "volume of 1024. It is cooled by a custom dilution refrigerator designed in-house.",

    "SkyScout drones have a flight time of 4 hours and carry a multispectral imaging "
    "payload. They use edge AI inference powered by NovaTech's NeuroCore chip for "
    "on-device crop analysis.",

    "DataPrime's data lake is built on Apache Iceberg and supports both batch and "
    "streaming workloads. After the NovaTech acquisition, it was rebranded as "
    "NovaTech DataCloud.",

    "GreenGrid's smart grid controller uses reinforcement learning to balance energy "
    "loads across distributed solar installations. The RL model was trained using "
    "NovaTech's cloud GPU cluster.",
]

print(f"Knowledge base: {len(DOCUMENTS)} passages")
for i, d in enumerate(DOCUMENTS):
    print(f"  [{i:2d}] {d[:80]}…")

## 5 — Knowledge Graph Construction

### 5.1 Entity & Relationship Extraction via LLM

Instead of relying on a pre-trained NER model (which may miss domain-specific entities),
we prompt our LLM to extract **structured triples** directly from each passage.

The extraction prompt asks the model to output lines in the format:
```
ENTITY: <name> | <type>
RELATION: <subject> | <predicate> | <object>
```

This is a form of **zero-shot information extraction** — no fine-tuning, no labelled
training data. The LLM's broad world knowledge lets it identify entities and relations
across diverse domains.

In [ ]:
import re
import json

EXTRACT_PROMPT = """Extract all named entities and relationships from the text below.

Output format — one item per line, no extra commentary:
ENTITY: <name> | <type>
RELATION: <subject> | <predicate> | <object>

Entity types: PERSON, ORG, PRODUCT, TECHNOLOGY, LOCATION, EVENT, DATE
Use short, canonical names (e.g., "NovaTech" not "the NovaTech company").

Text:
{text}
"""

def extract_entities_and_relations(text):
    """Use the LLM to pull entities and triples from a single passage."""
    raw = generate(EXTRACT_PROMPT.format(text=text), max_new_tokens=400, temperature=0.3)
    entities, relations = [], []
    for line in raw.splitlines():
        line = line.strip()
        if line.upper().startswith("ENTITY:"):
            parts = line[len("ENTITY:"):].split("|")
            if len(parts) >= 2:
                entities.append({"name": parts[0].strip(), "type": parts[1].strip()})
        elif line.upper().startswith("RELATION:"):
            parts = line[len("RELATION:"):].split("|")
            if len(parts) >= 3:
                relations.append({
                    "subject": parts[0].strip(),
                    "predicate": parts[1].strip(),
                    "object": parts[2].strip(),
                })
    return entities, relations

# Test on the first passage
ents, rels = extract_entities_and_relations(DOCUMENTS[0])
print("Entities:", json.dumps(ents, indent=2))
print("\nRelations:", json.dumps(rels, indent=2))

### 5.2 Extract From All Passages

We iterate over every document, collecting all entities and relationship triples.
We also track which passage each triple came from — this *provenance* link is
crucial for the retrieval step later.

In [ ]:
all_entities = {}   # name → type
all_relations = []  # list of (subj, pred, obj, source_idx)

for idx, doc in enumerate(DOCUMENTS):
    ents, rels = extract_entities_and_relations(doc)
    for e in ents:
        all_entities[e["name"]] = e["type"]
    for r in rels:
        all_relations.append((r["subject"], r["predicate"], r["object"], idx))
    print(f"[{idx:2d}] {len(ents)} entities, {len(rels)} relations")

print(f"\nTotal unique entities : {len(all_entities)}")
print(f"Total relation triples: {len(all_relations)}")

### 5.3 Build the Knowledge Graph with NetworkX

We convert the extracted triples into a `networkx.DiGraph`. Each node is an entity,
each directed edge carries a `predicate` label and the `source` passage index.

In [ ]:
import networkx as nx

KG = nx.DiGraph()

# Add entity nodes with type attribute
for name, etype in all_entities.items():
    KG.add_node(name, entity_type=etype)

# Add relation edges
for subj, pred, obj, src in all_relations:
    # Ensure nodes exist even if extraction was inconsistent
    if subj not in KG:
        KG.add_node(subj, entity_type="UNKNOWN")
    if obj not in KG:
        KG.add_node(obj, entity_type="UNKNOWN")
    KG.add_edge(subj, obj, predicate=pred, source=src)

print(f"Knowledge Graph: {KG.number_of_nodes()} nodes, {KG.number_of_edges()} edges")
print(f"\nSample nodes: {list(KG.nodes(data=True))[:5]}")
print(f"\nSample edges: {list(KG.edges(data=True))[:5]}")

### 5.4 Visualise the Knowledge Graph

Colour nodes by entity type, draw edge labels to show predicates.
This gives us a bird's-eye view of how entities relate.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

TYPE_COLORS = {
    "PERSON": "#FF6B6B", "ORG": "#4ECDC4", "PRODUCT": "#45B7D1",
    "TECHNOLOGY": "#96CEB4", "LOCATION": "#FFEAA7", "EVENT": "#DDA0DD",
    "DATE": "#FFB347", "UNKNOWN": "#C0C0C0",
}

node_colors = [TYPE_COLORS.get(KG.nodes[n].get("entity_type", "UNKNOWN"), "#C0C0C0")
               for n in KG.nodes]

plt.figure(figsize=(16, 11))
pos = nx.spring_layout(KG, k=2.5, seed=42)
nx.draw_networkx_nodes(KG, pos, node_color=node_colors, node_size=700, alpha=0.9)
nx.draw_networkx_labels(KG, pos, font_size=7, font_weight="bold")
nx.draw_networkx_edges(KG, pos, edge_color="#888888", arrows=True,
                       arrowsize=15, connectionstyle="arc3,rad=0.1")

# Edge labels (predicates)
edge_labels = {(u, v): d["predicate"] for u, v, d in KG.edges(data=True)}
nx.draw_networkx_edge_labels(KG, pos, edge_labels=edge_labels, font_size=5,
                             font_color="#555555")

# Legend
handles = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()
           if t in {KG.nodes[n].get("entity_type") for n in KG.nodes}]
plt.legend(handles=handles, loc="upper left", fontsize=8)
plt.title("Knowledge Graph — Extracted from Synthetic Corpus", fontsize=14)
plt.tight_layout()
plt.show()
print(f"Plotted {KG.number_of_nodes()} nodes, {KG.number_of_edges()} edges.")

## 6 — Graph-Based Retrieval

Given a query, we:
1. **Identify query entities** — find which KG nodes appear in (or are close to) the query.
2. **Extract a subgraph** — BFS/DFS around those nodes up to depth *k*.
3. **Collect source passages** linked to the edges in the subgraph.

This is fundamentally different from vector search: instead of "find chunks that
*sound* similar", we "find facts that are *structurally connected* to the query entities".

In [ ]:
def find_query_entities(query, kg, embedder, threshold=0.65):
    """
    Match query text to KG nodes using embedding similarity.
    Returns list of (node_name, similarity_score) above threshold.
    """
    import numpy as np
    query_emb = embedder.encode([query])
    node_names = list(kg.nodes)
    node_embs = embedder.encode(node_names)
    # Cosine similarity
    sims = np.dot(node_embs, query_emb.T).flatten()
    norms = np.linalg.norm(node_embs, axis=1) * np.linalg.norm(query_emb)
    sims = sims / (norms + 1e-10)
    matches = [(node_names[i], float(sims[i])) for i in range(len(sims)) if sims[i] >= threshold]
    matches.sort(key=lambda x: -x[1])
    return matches

def extract_subgraph(kg, seed_nodes, max_hops=2):
    """
    BFS from seed_nodes up to max_hops. Returns a subgraph and the
    set of source passage indices found on traversed edges.
    """
    visited = set()
    frontier = set(seed_nodes)
    for _ in range(max_hops):
        next_frontier = set()
        for node in frontier:
            if node not in kg:
                continue
            visited.add(node)
            for neighbor in list(kg.successors(node)) + list(kg.predecessors(node)):
                if neighbor not in visited:
                    next_frontier.add(neighbor)
        frontier = next_frontier
    visited |= frontier
    sub = kg.subgraph(visited).copy()
    sources = {d["source"] for _, _, d in sub.edges(data=True) if "source" in d}
    return sub, sources

# Demo: graph retrieval for a multi-hop question
query = "Who is the CEO of the company that acquired DataPrime?"
matches = find_query_entities(query, KG, embedder, threshold=0.45)
print(f"Query: {query}")
print(f"Matched entities: {matches}")

seed_names = [m[0] for m in matches[:5]]
subgraph, source_ids = extract_subgraph(KG, seed_names, max_hops=2)
print(f"\nSubgraph: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges")
print(f"Source passage indices: {sorted(source_ids)}")

In [ ]:
# Visualise the retrieved subgraph
plt.figure(figsize=(12, 8))
sub_colors = [TYPE_COLORS.get(subgraph.nodes[n].get("entity_type", "UNKNOWN"), "#C0C0C0")
              for n in subgraph.nodes]
pos_sub = nx.spring_layout(subgraph, k=2.0, seed=42)

# Highlight seed nodes
seed_set = set(seed_names)
node_borders = ["red" if n in seed_set else "black" for n in subgraph.nodes]
linewidths = [3 if n in seed_set else 1 for n in subgraph.nodes]

nx.draw_networkx_nodes(subgraph, pos_sub, node_color=sub_colors, node_size=800,
                       edgecolors=node_borders, linewidths=linewidths, alpha=0.9)
nx.draw_networkx_labels(subgraph, pos_sub, font_size=8, font_weight="bold")
nx.draw_networkx_edges(subgraph, pos_sub, edge_color="#888", arrows=True, arrowsize=15)
sub_labels = {(u, v): d["predicate"] for u, v, d in subgraph.edges(data=True)}
nx.draw_networkx_edge_labels(subgraph, pos_sub, edge_labels=sub_labels, font_size=7)

plt.title(f"Subgraph for: \"{query}\"", fontsize=12)
plt.tight_layout()
plt.show()
print("Red-bordered nodes = seed entities from the query.")

## 7 — Hybrid Retrieval: Graph + Vector

Graph traversal is great for **structural** queries ("Who acquired whom?"), but it
depends on accurate entity matching. Vector search is great for **semantic** queries
("Tell me about renewable energy innovations") but misses structural chains.

**Hybrid retrieval** combines both:
1. **Vector search** → retrieve top-k passages by semantic similarity.
2. **Graph search** → find query entities, traverse, collect linked passages.
3. **Merge & deduplicate** the two result sets.

This gives us the best of both worlds.

In [ ]:
import faiss
import numpy as np

# Embed all passages
doc_embeddings = embedder.encode(DOCUMENTS, normalize_embeddings=True)
dim = doc_embeddings.shape[1]

# Build FAISS index (Inner Product = cosine on normalised vectors)
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings.astype(np.float32))

print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")

In [ ]:
def vector_retrieve(query, k=5):
    """Return top-k passage indices by cosine similarity."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    return list(zip(indices[0].tolist(), scores[0].tolist()))

# Demo
v_results = vector_retrieve("renewable energy technology", k=5)
print("Vector-only results for 'renewable energy technology':")
for idx, score in v_results:
    print(f"  [{idx:2d}] score={score:.4f}  {DOCUMENTS[idx][:70]}…")

In [ ]:
def hybrid_retrieve(query, kg, embedder, index, documents, k_vector=5,
                    max_hops=2, entity_threshold=0.45, alpha=0.5):
    """
    Hybrid retrieval combining vector similarity and graph traversal.

    alpha controls the blend:  1.0 = vector only,  0.0 = graph only.
    Returns: list of (passage_idx, combined_score, source_label)
    """
    # ── Vector branch ─────────────────────────────────────────────
    vec_hits = vector_retrieve(query, k=k_vector)
    vec_scores = {idx: score for idx, score in vec_hits}

    # ── Graph branch ──────────────────────────────────────────────
    matches = find_query_entities(query, kg, embedder, threshold=entity_threshold)
    seed_names = [m[0] for m in matches[:5]]
    _, source_ids = extract_subgraph(kg, seed_names, max_hops=max_hops)
    graph_scores = {idx: 1.0 for idx in source_ids}  # binary signal from graph

    # ── Merge ─────────────────────────────────────────────────────
    all_ids = set(vec_scores.keys()) | set(graph_scores.keys())
    results = []
    for pid in all_ids:
        vs = vec_scores.get(pid, 0.0)
        gs = graph_scores.get(pid, 0.0)
        combined = alpha * vs + (1 - alpha) * gs
        label = []
        if pid in vec_scores: label.append("vector")
        if pid in graph_scores: label.append("graph")
        results.append((pid, combined, "+".join(label)))
    results.sort(key=lambda x: -x[1])
    return results

# Demo
h_results = hybrid_retrieve(
    "Who is the CEO of the company that acquired DataPrime?",
    KG, embedder, index, DOCUMENTS
)
print("Hybrid results:")
for pid, score, label in h_results[:8]:
    print(f"  [{pid:2d}] score={score:.4f} ({label})  {DOCUMENTS[pid][:65]}…")

## 8 — Multi-Hop Reasoning

Multi-hop questions require chaining multiple facts:

> *"Who is the CEO of the company that acquired DataPrime?"*

This requires two hops:
1. **Hop 1:** DataPrime was acquired by → NovaTech
2. **Hop 2:** NovaTech's CEO is → Dr. Elena Marchetti

Vector search alone would need both facts in the same chunk (or get lucky with
top-k overlap). The knowledge graph trivially chains them via graph traversal.

Let's implement explicit multi-hop path finding.

In [ ]:
def find_paths(kg, source, target, max_length=4):
    """
    Find all simple paths from source to target in the KG,
    with edge predicates, up to max_length hops.
    """
    paths_with_preds = []
    # Use underlying undirected view for reachability
    ug = kg.to_undirected()
    try:
        for path in nx.all_simple_paths(ug, source, target, cutoff=max_length):
            edges_info = []
            for i in range(len(path) - 1):
                u, v = path[i], path[i+1]
                if kg.has_edge(u, v):
                    pred = kg.edges[u, v]["predicate"]
                    edges_info.append(f"{u} --[{pred}]--> {v}")
                elif kg.has_edge(v, u):
                    pred = kg.edges[v, u]["predicate"]
                    edges_info.append(f"{v} --[{pred}]--> {u}")
                else:
                    edges_info.append(f"{u} -- {v}")
            paths_with_preds.append((path, edges_info))
    except nx.NetworkXError:
        pass
    return paths_with_preds

# Demo: find chain from DataPrime to Dr. Elena Marchetti
paths = find_paths(KG, "DataPrime", "Dr. Elena Marchetti", max_length=4)
print(f"Paths from DataPrime → Dr. Elena Marchetti ({len(paths)} found):\n")
for i, (path, edges) in enumerate(paths):
    print(f"  Path {i+1}: {' → '.join(path)}")
    for e in edges:
        print(f"    {e}")
    print()

In [ ]:
def graph_multihop_context(query, kg, embedder, max_hops=3):
    """
    Build a textual context from graph paths for multi-hop questions.
    1. Identify query entities.
    2. For each pair of query entities, find shortest paths.
    3. Collect edge triples along those paths as context.
    """
    matches = find_query_entities(query, kg, embedder, threshold=0.40)
    seeds = [m[0] for m in matches[:6]]
    context_lines = []
    seen_edges = set()
    for i, s in enumerate(seeds):
        for t in seeds[i+1:]:
            for path, edges in find_paths(kg, s, t, max_length=max_hops):
                for e in edges:
                    if e not in seen_edges:
                        context_lines.append(f"- {e}")
                        seen_edges.add(e)
    # Also collect direct edges from seed nodes
    for s in seeds:
        if s in kg:
            for _, nbr, d in kg.edges(s, data=True):
                line = f"- {s} --[{d['predicate']}]--> {nbr}"
                if line not in seen_edges:
                    context_lines.append(line)
                    seen_edges.add(line)
            for nbr, _, d in kg.in_edges(s, data=True):
                line = f"- {nbr} --[{d['predicate']}]--> {s}"
                if line not in seen_edges:
                    context_lines.append(line)
                    seen_edges.add(line)
    return "\n".join(context_lines)

q = "Who is the CEO of the company that acquired DataPrime?"
graph_ctx = graph_multihop_context(q, KG, embedder)
print(f"Query: {q}\n")
print(f"Graph context ({len(graph_ctx.splitlines())} triples):")
print(graph_ctx)

In [ ]:
ANSWER_PROMPT = """Use ONLY the context below to answer the question.
If the context does not contain enough information, say so.

Context:
{context}

Question: {question}

Answer:"""

answer = generate(ANSWER_PROMPT.format(context=graph_ctx, question=q), max_new_tokens=150)
print(f"Q: {q}")
print(f"A: {answer}")

## 9 — Community Detection

Large knowledge graphs naturally contain **clusters** of densely interconnected
entities. Community detection algorithms identify these clusters automatically.

Why does this matter for RAG?
- **Summarisation:** Generate a summary per community to create a hierarchical index.
- **Scoping:** When a query matches entities in one community, prioritise passages
  from that community.
- **Exploration:** Help users understand the thematic structure of a corpus.

We use the **Louvain** method (greedy modularity optimisation) on the undirected
projection of our KG.

In [ ]:
# Convert to undirected for community detection
UG = KG.to_undirected()

# Louvain community detection
communities = nx.community.louvain_communities(UG, seed=42)

print(f"Found {len(communities)} communities:\n")
for i, comm in enumerate(communities):
    members = sorted(comm)
    print(f"  Community {i+1} ({len(members)} members): {members}")

In [ ]:
# Visualise communities with distinct colours
COMM_COLORS = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7",
               "#DDA0DD", "#FFB347", "#87CEEB", "#98D8C8"]

node_to_comm = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_to_comm[node] = i

comm_node_colors = [COMM_COLORS[node_to_comm.get(n, 0) % len(COMM_COLORS)] for n in KG.nodes]

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(KG, k=2.5, seed=42)
nx.draw_networkx_nodes(KG, pos, node_color=comm_node_colors, node_size=700, alpha=0.9)
nx.draw_networkx_labels(KG, pos, font_size=7, font_weight="bold")
nx.draw_networkx_edges(KG, pos, edge_color="#888", arrows=True, arrowsize=12,
                       connectionstyle="arc3,rad=0.1")

handles = [mpatches.Patch(color=COMM_COLORS[i], label=f"Community {i+1}")
           for i in range(len(communities))]
plt.legend(handles=handles, loc="upper left", fontsize=9)
plt.title("Knowledge Graph — Community Structure (Louvain)", fontsize=14)
plt.tight_layout()
plt.show()
print(f"Modularity: {nx.community.modularity(UG, communities):.4f}")

In [ ]:
def summarise_community(kg, community_nodes, documents):
    """Collect all source passages for edges within a community and summarise."""
    source_ids = set()
    sub = kg.subgraph(community_nodes)
    for u, v, d in sub.edges(data=True):
        if "source" in d:
            source_ids.add(d["source"])
    # Also add edges from/to community nodes in the full graph
    for node in community_nodes:
        for _, _, d in kg.edges(node, data=True):
            if "source" in d:
                source_ids.add(d["source"])
    passages = [documents[i] for i in sorted(source_ids) if i < len(documents)]
    if not passages:
        return "No source passages found."
    text = "\n".join(passages)
    prompt = f"Summarise the following information in 2-3 sentences:\n\n{text}"
    return generate(prompt, max_new_tokens=120, temperature=0.3)

print("Community Summaries:\n" + "="*60)
for i, comm in enumerate(communities):
    summary = summarise_community(KG, comm, DOCUMENTS)
    print(f"\nCommunity {i+1} ({sorted(comm)[:4]}…):")
    print(f"  {summary}")

## 10 — Complete Graph-RAG Pipeline

We now wire everything together into a single function that:
1. Performs **hybrid retrieval** (vector + graph).
2. Optionally adds **graph path context** for multi-hop queries.
3. Constructs a prompt and generates the final answer.

This is the full end-to-end pipeline.

In [ ]:
def graph_rag(query, kg, embedder, index, documents, k_vector=5,
              max_hops=2, alpha=0.5, use_paths=True):
    """
    Complete Graph-RAG pipeline.

    Returns: (answer, context_str, retrieval_results)
    """
    # ── Hybrid retrieval ─────────────────────────────────────────
    results = hybrid_retrieve(query, kg, embedder, index, documents,
                              k_vector=k_vector, max_hops=max_hops, alpha=alpha)
    top_ids = [r[0] for r in results[:8]]
    passage_context = "\n\n".join(f"[Passage {i}] {documents[i]}" for i in top_ids)

    # ── Graph path context (multi-hop) ───────────────────────────
    path_context = ""
    if use_paths:
        path_context = graph_multihop_context(query, kg, embedder, max_hops=max_hops+1)
        if path_context:
            path_context = f"\n\nKnowledge Graph relationships:\n{path_context}"

    # ── Generate ─────────────────────────────────────────────────
    full_context = passage_context + path_context
    prompt = ANSWER_PROMPT.format(context=full_context, question=query)
    answer = generate(prompt, max_new_tokens=200)
    return answer, full_context, results

# Run the full pipeline
test_q = "Who is the CEO of the company that acquired DataPrime?"
ans, ctx, res = graph_rag(test_q, KG, embedder, index, DOCUMENTS)
print(f"Q: {test_q}")
print(f"A: {ans}")
print(f"\nRetrieved {len(res)} passages (top 5 shown):")
for pid, score, label in res[:5]:
    print(f"  [{pid:2d}] {score:.3f} ({label})")

### 10.1 Test on Diverse Queries

Let's test the full pipeline on several query types:
- **Factual:** direct entity lookup.
- **Relational:** requires understanding a specific relationship.
- **Multi-hop:** requires chaining 2+ facts.
- **Semantic:** broad thematic question.

In [ ]:
TEST_QUERIES = [
    # Factual
    "Where is QuantumLeap headquartered?",
    # Relational
    "What is the partnership between NovaTech and QuantumLeap about?",
    # Multi-hop
    "What chip does the SkyScout drone use, and who manufactures it?",
    # Semantic
    "How is AI being applied in agriculture and energy sectors?",
]

for q in TEST_QUERIES:
    ans, _, _ = graph_rag(q, KG, embedder, index, DOCUMENTS)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print("-" * 70)

## 11 — Comparison: Vector-Only vs. Graph-Augmented RAG

To appreciate the benefit of the knowledge graph, let's build a **vector-only**
RAG pipeline and compare the two side by side on our multi-hop test queries.

In [ ]:
def vector_only_rag(query, index, documents, k=5):
    """Baseline: vector-only RAG with no graph."""
    hits = vector_retrieve(query, k=k)
    top_ids = [idx for idx, _ in hits]
    context = "\n\n".join(f"[Passage {i}] {documents[i]}" for i in top_ids)
    prompt = ANSWER_PROMPT.format(context=context, question=query)
    answer = generate(prompt, max_new_tokens=200)
    return answer, top_ids

COMPARISON_QUERIES = [
    "Who is the CEO of the company that acquired DataPrime?",
    "What chip does the SkyScout drone use, and who manufactures it?",
    "What role does Sarah Chen hold after the acquisition, and who does she report to?",
]

print("=" * 80)
print("VECTOR-ONLY vs GRAPH-RAG COMPARISON")
print("=" * 80)

for q in COMPARISON_QUERIES:
    print(f"\nQ: {q}")
    print("-" * 60)

    v_ans, v_ids = vector_only_rag(q, index, DOCUMENTS)
    print(f"  [Vector-Only] passages={v_ids}")
    print(f"  A: {v_ans}")

    g_ans, _, g_res = graph_rag(q, KG, embedder, index, DOCUMENTS)
    g_ids = [r[0] for r in g_res[:5]]
    print(f"\n  [Graph-RAG]   passages={g_ids}")
    print(f"  A: {g_ans}")
    print("=" * 80)

## 12 — Graph Analytics

Let's examine the structural properties of our knowledge graph. These metrics
help us understand the quality and coverage of the extracted graph.

In [ ]:
print("Knowledge Graph Statistics")
print("=" * 50)
print(f"Nodes             : {KG.number_of_nodes()}")
print(f"Edges             : {KG.number_of_edges()}")
print(f"Density           : {nx.density(KG):.4f}")
print(f"Connected components (undirected): {nx.number_connected_components(UG)}")
print(f"Average clustering (undirected)  : {nx.average_clustering(UG):.4f}")

# Degree centrality — which entities are most connected?
deg_cent = nx.degree_centrality(KG)
top_nodes = sorted(deg_cent.items(), key=lambda x: -x[1])[:10]
print(f"\nTop entities by degree centrality:")
for name, cent in top_nodes:
    etype = KG.nodes[name].get('entity_type', '?')
    print(f"  {name:30s} ({etype:10s})  centrality={cent:.4f}")

# Betweenness centrality — bridge entities
bet_cent = nx.betweenness_centrality(KG)
top_bridge = sorted(bet_cent.items(), key=lambda x: -x[1])[:5]
print(f"\nTop bridge entities (betweenness centrality):")
for name, cent in top_bridge:
    print(f"  {name:30s}  betweenness={cent:.4f}")

In [ ]:
from collections import Counter

type_counts = Counter(d.get("entity_type", "UNKNOWN") for _, d in KG.nodes(data=True))
print("Entity type distribution:")
for etype, count in type_counts.most_common():
    bar = "█" * count
    print(f"  {etype:12s} {count:3d}  {bar}")

pred_counts = Counter(d["predicate"] for _, _, d in KG.edges(data=True))
print(f"\nRelationship predicate distribution:")
for pred, count in pred_counts.most_common(10):
    bar = "█" * count
    print(f"  {pred:30s} {count:3d}  {bar}")

## 13 — Retrieval Quality Analysis

Let's quantify how much the graph helps by checking whether each method retrieves
the **gold passages** (passages that contain the answer) for our test queries.

In [ ]:
# Ground truth: which passages are needed for each query?
GOLD = {
    "Who is the CEO of the company that acquired DataPrime?": {0, 5},    # NovaTech CEO + acquisition
    "What chip does the SkyScout drone use, and who manufactures it?": {10, 12},  # NeuroCore + SkyScout
    "What role does Sarah Chen hold after the acquisition, and who does she report to?": {9, 5},  # Sarah + acquisition
}

def recall_at_k(retrieved_ids, gold_ids, k=5):
    retrieved_set = set(retrieved_ids[:k])
    return len(retrieved_set & gold_ids) / len(gold_ids)

print(f"{'Query':<65s} {'Vec R@5':>8s} {'Graph R@5':>10s}")
print("-" * 85)

for q, gold in GOLD.items():
    v_hits = vector_retrieve(q, k=5)
    v_ids = [idx for idx, _ in v_hits]
    v_recall = recall_at_k(v_ids, gold, k=5)

    h_results = hybrid_retrieve(q, KG, embedder, index, DOCUMENTS)
    g_ids = [r[0] for r in h_results[:5]]
    g_recall = recall_at_k(g_ids, gold, k=5)

    short_q = q[:60] + "…" if len(q) > 60 else q
    print(f"{short_q:<65s} {v_recall:>8.2f} {g_recall:>10.2f}")

## 14 — Synthesis: When Does Graph-RAG Help?

### ✅ Graph-RAG shines when:

| Scenario | Why the graph helps |
|---|---|
| **Multi-hop questions** | Path traversal chains facts that live in different chunks |
| **Structured domains** (legal, medical, finance) | Entities and relations are well-defined |
| **Relational queries** ("Who reports to whom?") | Edges encode exactly this |
| **Small-to-medium corpora** with dense entity networks | Graph construction is tractable |
| **Explainability required** | The graph path *is* the explanation |

### ⚠️ Caveats:

| Concern | Detail |
|---|---|
| **Construction cost** | Extracting entities/relations requires an LLM call *per passage* |
| **Extraction quality** | The KG is only as good as the extraction prompt |
| **Maintenance** | When documents change, the graph must be updated |
| **Scalability** | Very large graphs (millions of nodes) need dedicated graph DBs (Neo4j, etc.) |
| **Overkill for simple queries** | "What is photosynthesis?" doesn't need a graph |

### Design Guidelines

1. **Start with vector RAG.** It's simpler and covers most queries well.
2. **Add a graph layer** when you see failures on relational/multi-hop questions.
3. **Hybrid retrieval** (alpha-blend) lets you tune the balance per use case.
4. **Community summaries** create a hierarchical index that helps with global queries.
5. **Persist the graph** (e.g., in Neo4j or as a serialised NetworkX pickle) so you
   don't re-extract on every run.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** swap out the retriever/chunker and measure the impact on recall. Document what changes and why.

**Exercise 2 — Build:** add a new document type and test retrieval quality. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** implement a simple version of the algorithm from scratch.

## 15 — Key Takeaways

1. **Vector search is semantic but structurally blind.** It finds what *sounds* related
   but cannot chain facts across chunks.

2. **Knowledge graphs capture structure.** Nodes = entities, edges = relationships,
   triples = the atomic unit `(subject, predicate, object)`.

3. **LLMs can extract KGs from text** with zero-shot prompting — no NER library needed.

4. **Graph-based retrieval** identifies query entities, traverses the graph (BFS),
   and collects structurally relevant passages.

5. **Hybrid retrieval** (vector + graph) combines semantic similarity with structural
   reasoning via an alpha-weighted merge.

6. **Multi-hop reasoning** follows paths in the graph to answer questions that span
   multiple passages (`A → acquired → B → CEO → C`).

7. **Community detection** reveals entity clusters and enables hierarchical
   summarisation of the corpus.

8. **Graph-RAG is most valuable** for structured domains, relational queries, and
   multi-hop reasoning. For simple semantic queries, vector-only RAG may suffice.

9. **The trade-off** is construction cost (LLM calls per passage) and maintenance
   (graph must be updated when documents change).

10. **Start simple, add graphs when needed.** The hybrid approach lets you dial
    the graph contribution up or down with a single parameter.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the rag/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [Fusion Retrieval](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/fusion_retrieval.ipynb) | ➡️ **Next:** [Hierarchical Indices](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/hierarchical_indices.ipynb)